## MODFLOW 6 Lab: Customizing for Flow, Transport and Geothermal Modeling

### 1.1 Hands-on: Flow Model 

In this session, you'll learn to:
- Build a simple groundwater flow model using MODFLOW 6.
- Visualize flow and results 
- Extend the model with dynamic pumping and river interactions.

We'll work step-by-step and feel free to modify parameters and rerun cells!

### Model description 

This groundwater model simulates the extraction of water of an aquifer.
#### Domain 
The model domain is a 2D single-layer grid consisting of 101 rows and columns, representing a 100 m × 100 m unconfined aquifer with a thickness of 30 m. 
#### Boundary conditions 
Boundary conditions include constant head boundaries on the left and right edges of the domain to simulate regional flow, with head values set to 25.0 m and 24.5 m, respectively. One extraction well is position in the top of the aquifer at cell x. 

### Goal description 

Regulate the model to extract the most amount of water possible while being regulated by the groundwater level threshold of 1 m. 

### Setup and Imports 

In [169]:
import flopy
import numpy as np
import matplotlib.pyplot as plt
import os

# Define workspace
workspace = os.path.join(os.getcwd(), "models/mf6/ws_example_1")
os.makedirs(workspace, exist_ok=True)
print("Workspace:", workspace)

Workspace: C:\Users\lucialabarca\re-run noteboks\pymf6-validation\src\notebooks\presentation_files\models/mf6/ws_example_1


### Build the MODFLOW 6 Base Flow Model 

## Start setting up the model with pymf6 

### Magic commands - auto reload of the model each time

In [170]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [171]:
%autoreload 2

### Import from pymf6tools the functions to run, get and visualize simulation results

In [172]:
from pathlib import Path 
from pymf6.mf6 import MF6
import pandas as pd 
from functools import partial 
import numpy as np 

from pymf6_tools.make_model import run_simulation, get_simulation
from pymf6_tools.base_model import make_model_data
from pymf6_tools.make_model import make_input, run_simulation, get_simulation
from pymf6_tools.plotting import show_heads, show_well_head, show_bcs

In [173]:
from pymf6_tools.plotting import (
show_heads, show_well_head, show_concentration, show_bcs, 
show_bot_elevations, show_river_stages, contour_bot_elevations, 
plot_spec_discharge)
import pymf6_tools

## Setup model path and name 

In [174]:
model_path = 'models/mf6/ws_example_1'
model_name = "ws_example_1"

### Run simulation

In [175]:
run_simulation(model_path, verbosity_level=0)

### Inpect model parameters 

In [176]:
sim = get_simulation(model_path, model_name)
ml = sim.get_model('gwf_' + model_name)
dis = ml.get_package('dis') 

In [177]:
dis.data_list

[,
 ,
 ,
 ,
 ,
 ,
 ,
 ,
 ,
 {internal}
 (1),
 {internal}
 (101),
 {internal}
 (101),
 {constant 1.0},
 {constant 1.0},
 {constant 0.0},
 {constant -30.0},
 ]

## Visualization of Input and Output - e.g. Boundary conditions, Heads and Contamination plume¶

#### Boundary Conditions
Note that you should change the "bc_names" according to the boundary conditions present in the simulation. For instance:
'chd' Constant-head boundary
'riv-1' River boundary

In [178]:
show_bcs?

Signature:
show_bcs(
    model_path,
    name,
    title='Boundary Conditions',
    bc_names=('chd', 'wel', 'riv'),
    show_grid=True,
)
Docstring: Show location of boundary conditions.
File:      c:\users\lucialabarca\re-run noteboks\pymf6-validation\.pixi\envs\default\lib\site-packages\pymf6_tools\plotting.py
Type:      function

### Groundwater level 

In [179]:
#show_heads(model_path, model_name, show_wells=False, kstpkper=(19, 1), show_grid=False)

This graph shows the groundwater head distribution in the domain. 

🔍 What is kstpkper?
In MODFLOW 6, kstpkper = (kstp, kper) refers to:

kstp: Time step within a stress period

kper: Stress period number

FloPy tracks head and budget data per (kstp, kper). If you ask for a timestep combination that wasn't saved or doesn't exist, it throws this error.

✅ How to Find your kstper?
Check your simulation's stress period setup

From the model in this case 

perioddata=[(1.0, 1, 1.0), (300.0, 20, 1.0)]

There are:

Stress period 0: 1 time step

Stress period 1: 20 time steps

So valid kstpkper values:

For kper=0: only (0, 0)

For kper=1: (0, 1) to (19, 1)

Since we want the final time then is its (19, 1).

### Inpect MODFLOW 6 interactively 

pymf6 allows to access all MODFLOW 6 variables at run time. First, we import the class MF6:

In [180]:
from pymf6.mf6 import MF6

We use the model path 

In [181]:
sim_path = 'models/pymf6/ws_example_1'

In [ ]:
mf6 = MF6(sim_path=sim_path)


The instance has an HTML representation with some meta data:

In [ ]:
mf6

In [ ]:
mf6.info 

In [ ]:
mf6.simulation

In [ ]:
mf6.simulation.model_names

The meta information is also available as a dictionary:

In [ ]:
mf6.simulation.models_meta 

This example has only one solution group:

In [ ]:
mf6.simulation.solution_groups 

Time discretization:

In [ ]:
mf6.simulation.TDIS

The names of the variables

In [ ]:
mf6.simulation.TDIS.var_names

The scalar values 

In [ ]:
mf6.simulation.TDIS.NPER

and arrays:

In [ ]:
mf6.simulation.TDIS.PERLEN

The docstrings are extracted from the MODFLOW 6 source code from the used MODFLOW version as displayed with mf6.info.

The instance has many variables:

In [ ]:
len(mf6.vars)

Filter for all TDIS entries:

In [ ]:
{k: v for k, v in mf6.vars.items() if 'TDIS' in k}

Show the first 20:

In [ ]:
dict(list(mf6.vars.items())[:20])

A simulation can have several models types. These include:

+ gwf6 - flow model

+ gwt6 - constituent transport model

+ gwe6 - energy transport model

They are available as a dictionary:

In [ ]:
mf6.models.keys()

The example has only a flow models. Let’s get them:

In [ ]:
flow_models = mf6.models['gwf6']

MODFLOW supports multiple flow models. An example is the nesting an inner model with finer discretization into an outer model withcoarse discritization. Let’s get the flow model names:

In [ ]:
flow_models.keys()

This simple simulation has only one flow model. We get a reference to this model:

In [ ]:
gwf = flow_models['gwf_ws_example_1']

We can get a list if all available attributes, removing all names beginning with an underscore because they are internal names or special methods:

In [ ]:
[attr for attr in dir(gwf) if not attr.startswith('_')]

All attributes are available via tab completion, i.e. typing <TAB> after the dot will show the list of available attributes and will narrow it down to match typed characters:

In [ ]:
gwf.nper

The packages of a model are also available:

In [ ]:
gwf.packages

The well boundary condition is potentially mutable. Let’s try to get a mutable version:

In [ ]:
gwf.packages.mywell.as_mutable_bc()

This doesn’t work yet, because there is no boundary condition in the first, steady-state, stress period.

We create a reference to the model loop:

In [ ]:
loop = mf6.model_loop()

and progress to the start of the second stress period:

In [ ]:
for model_step in loop:
    if gwf.kper > 0:
        break

Note: Remember that the stress period count is zero-based. Therefore, the stress period with the index 0 is the first.

Now, we are at the beginning of the second stress period:

In [ ]:
model_step.state

and can create a mutable version of our well boundary conditions:

In [ ]:
gwf.packages.chd_0.as_mutable_bc()

In [ ]:
mywell = gwf.packages.mywell.as_mutable_bc()

There is one well in the middle of the model.

In [ ]:
MF6?

We can progress to the next stress period:

In [ ]:
for model_step in loop:
    if gwf.kper > 1:
        break

In [ ]:
mywell

we can modify the pumping rate and make itr 10% higher: 

In [ ]:
mywell.q *= 1.1

The changes are written back into MODFLOW simulation and will be used until for the calculations until MODFLOW reads new values for this boundary condition. This happens at the start of the next stress period or when new time series values are read.

In [ ]:
mywell

In [ ]:
model_step.state

In [ ]:
model_step.simulation_group.model_names

In [ ]:
model_step.simulation_group.kper

In [ ]:
gwf